In [ ]:
 # ==========================================
# Basic Image Processing using PyTorch
# Histogram Equalization, Thresholding,
# Edge Detection, Data Augmentation,
# Morphological Operations
# ==========================================

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# --------------------------------
# 1. Load Image
# --------------------------------
image = Image.open("image.jpg").convert("L")

transform = transforms.ToTensor()

img = transform(image)

plt.imshow(img.squeeze(), cmap='gray')
plt.title("Original Image")
plt.axis("off")
plt.show()

# --------------------------------
# 2. Histogram Equalization
# --------------------------------

hist = torch.histc(img, bins=256, min=0, max=1)
cdf = torch.cumsum(hist, dim=0)
cdf = cdf / cdf[-1]

img_eq = torch.floor(img * 255).long()
img_eq = cdf[img_eq] / 255

plt.imshow(img_eq.squeeze(), cmap='gray')
plt.title("Histogram Equalization")
plt.axis("off")
plt.show()

# --------------------------------
# 3. Thresholding
# --------------------------------

threshold = 0.5
binary = (img > threshold).float()

plt.imshow(binary.squeeze(), cmap='gray')
plt.title("Thresholding")
plt.axis("off")
plt.show()

# --------------------------------
# 4. Edge Detection (Sobel)
# --------------------------------

sobel_x = torch.tensor([[-1,0,1],
                        [-2,0,2],
                        [-1,0,1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

sobel_y = torch.tensor([[-1,-2,-1],
                        [0,0,0],
                        [1,2,1]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)

img_batch = img.unsqueeze(0)

edge_x = F.conv2d(img_batch, sobel_x, padding=1)
edge_y = F.conv2d(img_batch, sobel_y, padding=1)

edges = torch.sqrt(edge_x**2 + edge_y**2)

plt.imshow(edges.squeeze().detach(), cmap='gray')
plt.title("Edge Detection (Sobel)")
plt.axis("off")
plt.show()

# --------------------------------
# 5. Data Augmentation
# --------------------------------

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1),
    transforms.RandomRotation(30)
])

augmented = augment(image)
augmented = transform(augmented)

plt.imshow(augmented.squeeze(), cmap='gray')
plt.title("Data Augmentation")
plt.axis("off")
plt.show()

# --------------------------------
# 6. Morphological Operations
# --------------------------------

kernel = torch.ones((1,1,3,3))

binary_batch = binary.unsqueeze(0)

# Dilation
dilated = F.max_pool2d(binary_batch, kernel_size=3, stride=1, padding=1)

plt.imshow(dilated.squeeze(), cmap='gray')
plt.title("Dilation")
plt.axis("off")
plt.show()

# Erosion
eroded = -F.max_pool2d(-binary_batch, kernel_size=3, stride=1, padding=1)

plt.imshow(eroded.squeeze(), cmap='gray')
plt.title("Erosion")
plt.axis("off")
plt.show()

